# 03 Which reader wins

**Question.** Does P1 (vector-located crops) beat P2 (whole page) on the same gold, and at what cost?

**Answer.** On all gold, P1 wins on the primary metric, F1: 0.725 against 0.650. The win comes from recall (0.862 against 0.672) at the same precision (0.625 against 0.629), and P1 also finds more of the partial gold (0.743 against 0.543).

P1 is not better everywhere:
- On the dev split alone, P2 has the higher F1 (0.757 against 0.739). Dev has only two drawings with full gold, CTC-01 and FTC-07 page 2, and P2 wins FTC-07. P1 still wins dev recall and partial recall.
- It loses on two of the four full-gold drawings, CTC-03 and FTC-07 page 2, both on precision.
- It is slower: 64 seconds per page against 36.

This notebook reads the runs of all three readers from the run store; it runs no model.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # notebooks run from experiments/drawing/
from notebook_helpers import (  # noqa: E402
    case_table,
    evals,
    pd,
    run_summary,
    stored_run,
)

## The runs

`textlayer-v0` reads only the title block and notes from the PDF text layer, so it shows what any reader gets for free. Each reader has one run on all 11 drawings and one on the dev split.

In [2]:
versions = ("textlayer-v0", "whole-page-v1", "vector-crop-v1")
runs = {version: stored_run(version) for version in versions}
dev = {version: stored_run(version, "dev") for version in versions}

## The three readers side by side

Every metric comes from the harness summary over all gold: F1, precision and recall over the four drawings with full gold, partial recall over the other seven. Seconds per page is the harness latency divided by the page count (FTC-07's four pages are all read, though only page 2 is scored).

In [3]:
pd.DataFrame([run_summary(run) for run in runs.values()]).set_index("version").round(3)

,run,F1,precision,recall,partial recall,errors,pages,minutes,seconds per page
version,,,,,,,,,
textlayer-v0,reader-20261102T043459Z-072d,0.344,0.902,0.213,0.129,0,30,0.0,0.0
whole-page-v1,reader-20261102T050825Z-2274,0.650,0.629,0.672,0.543,0,30,18.2,36.3
vector-crop-v1,reader-20261102T043630Z-7f02,0.725,0.625,0.862,0.743,0,30,31.9,63.8


## `evals compare`: P2 as the base, P1 as the candidate

In [4]:
evals("compare", "reader", runs["whole-page-v1"], runs["vector-crop-v1"])

base       reader-20261102T050825Z-2274  whole-page-v1  hash 7c7fecc1c3fe  split all  11 case(s)
candidate  reader-20261102T043630Z-7f02  vector-crop-v1  hash 48b6757c7e1d  split all  11 case(s)

metric          base   candidate  delta   verdict
f1 (primary)    0.650  0.725      +0.075  better
precision       0.629  0.625      -0.004  worse
recall          0.672  0.862      +0.190  better
partial_recall  0.543  0.743      +0.200  better

10 of 11 case(s) changed case_score
  nist_ctc_03_asme1_rc  0.757 -> 0.630  -0.126
  nist_ftc_07_asme1_rd  0.724 -> 0.603  -0.121
  nist_ftc_11_asme1_rb  0.700 -> 0.800  +0.100
  nist_ctc_01_asme1_rd  0.800 -> 0.957  +0.157
  nist_ctc_02_asme1_rc  0.500 -> 0.700  +0.200
  nist_ftc_09_asme1_rd  0.800 -> 1.000  +0.200
  nist_ftc_06_asme1_rd  0.546 -> 0.759  +0.212
  nist_ctc_05_asme1_rd  0.200 -> 0.500  +0.300
  nist_ctc_04_asme1_rd  0.600 -> 0.900  +0.300
  nist_ftc_08_asme1_rc  0.500 -> 0.800  +0.300



'base       reader-20261102T050825Z-2274  whole-page-v1  hash 7c7fecc1c3fe  split all  11 case(s)\ncandidate  reader-20261102T043630Z-7f02  vector-crop-v1  hash 48b6757c7e1d  split all  11 case(s)\n\nmetric          base   candidate  delta   verdict\nf1 (primary)    0.650  0.725      +0.075  better\nprecision       0.629  0.625      -0.004  worse\nrecall          0.672  0.862      +0.190  better\npartial_recall  0.543  0.743      +0.200  better\n\n10 of 11 case(s) changed case_score\n  nist_ctc_03_asme1_rc  0.757 -> 0.630  -0.126\n  nist_ftc_07_asme1_rd  0.724 -> 0.603  -0.121\n  nist_ftc_11_asme1_rb  0.700 -> 0.800  +0.100\n  nist_ctc_01_asme1_rd  0.800 -> 0.957  +0.157\n  nist_ctc_02_asme1_rc  0.500 -> 0.700  +0.200\n  nist_ftc_09_asme1_rd  0.800 -> 1.000  +0.200\n  nist_ftc_06_asme1_rd  0.546 -> 0.759  +0.212\n  nist_ctc_05_asme1_rd  0.200 -> 0.500  +0.300\n  nist_ctc_04_asme1_rd  0.600 -> 0.900  +0.300\n  nist_ftc_08_asme1_rc  0.500 -> 0.800  +0.300\n'

## Per drawing

In [5]:
pd.concat(
    {
        version: case_table(run)[["recall", "F1", "seconds per page"]]
        for version, run in runs.items()
    },
    axis=1,
)

textlayer-v0                         whole-page-v1  \
                           recall     F1 seconds per page        recall   
drawing                                                                   
nist_ctc_01_asme1_rd        0.261  0.414              0.0         0.783   
nist_ctc_02_asme1_rc        0.200    NaN              0.0         0.500   
nist_ctc_03_asme1_rc        0.189  0.318              0.0         0.757   
nist_ctc_04_asme1_rd        0.200    NaN              0.0         0.600   
nist_ctc_05_asme1_rd        0.100    NaN              0.0         0.200   
nist_ftc_06_asme1_rd        0.213  0.339              0.0         0.562   
nist_ftc_07_asme1_rd        0.200  0.333              0.0         0.840   
nist_ftc_08_asme1_rc        0.200    NaN              0.0         0.500   
nist_ftc_09_asme1_rd        0.100    NaN              0.0         0.800   
nist_ftc_10_asme1_rb        0.100    NaN              0.0         0.500   
nist_ftc_11_asme1_rb        0.000    NaN              0.0         0.700   

                                             vector-crop-v1         \
                         F1 seconds per page         recall     F1   
drawing                                                              
nist_ctc_01_asme1_rd  0.800             40.8          0.957  0.957   
nist_ctc_02_asme1_rc    NaN             35.0          0.700    NaN   
nist_ctc_03_asme1_rc  0.757             51.2          0.784  0.630   
nist_ctc_04_asme1_rd    NaN             32.7          0.900    NaN   
nist_ctc_05_asme1_rd    NaN             24.4          0.500    NaN   
nist_ftc_06_asme1_rd  0.546             47.2          0.865  0.759   
nist_ftc_07_asme1_rd  0.724             32.5          0.880  0.603   
nist_ftc_08_asme1_rc    NaN             41.7          0.800    NaN   
nist_ftc_09_asme1_rd    NaN             38.8          1.000    NaN   
nist_ftc_10_asme1_rb    NaN             33.5          0.500    NaN   
nist_ftc_11_asme1_rb    NaN             25.1          0.800    NaN   

                                       
                     seconds per page  
drawing                                
nist_ctc_01_asme1_rd             68.8  
nist_ctc_02_asme1_rc             62.7  
nist_ctc_03_asme1_rc            120.6  
nist_ctc_04_asme1_rd             91.4  
nist_ctc_05_asme1_rd             31.8  
nist_ftc_06_asme1_rd             72.9  
nist_ftc_07_asme1_rd             54.0  
nist_ftc_08_asme1_rc             65.1  
nist_ftc_09_asme1_rd             74.3  
nist_ftc_10_asme1_rb             68.8  
nist_ftc_11_asme1_rb             22.7

## The dev split

The same three readers on the five dev drawings only: CTC-01 and FTC-07 page 2 with full gold, CTC-02, CTC-04 and FTC-09 with partial gold.

In [6]:
pd.DataFrame([run_summary(run) for run in dev.values()]).set_index("version").round(3)

,run,F1,precision,recall,partial recall,errors,pages,minutes,seconds per page
version,,,,,,,,,
textlayer-v0,reader-20261102T043458Z-42fc,0.373,1.000,0.229,0.167,0,13,0.0,0.0
whole-page-v1,reader-20261102T054056Z-63e1,0.757,0.709,0.812,0.633,0,13,7.7,35.7
vector-crop-v1,reader-20261102T052636Z-4cb6,0.739,0.620,0.917,0.867,0,13,14.3,66.1


In [7]:
evals("compare", "reader", dev["whole-page-v1"], dev["vector-crop-v1"])

base       reader-20261102T054056Z-63e1  whole-page-v1  hash 7c7fecc1c3fe  split dev  5 case(s)
candidate  reader-20261102T052636Z-4cb6  vector-crop-v1  hash 48b6757c7e1d  split dev  5 case(s)

metric          base   candidate  delta   verdict
f1 (primary)    0.757  0.739      -0.018  worse
precision       0.709  0.620      -0.089  worse
recall          0.812  0.917      +0.104  better
partial_recall  0.633  0.867      +0.233  better

5 of 5 case(s) changed case_score
  nist_ftc_07_asme1_rd  0.724 -> 0.603  -0.121
  nist_ctc_01_asme1_rd  0.800 -> 0.957  +0.157
  nist_ctc_02_asme1_rc  0.500 -> 0.700  +0.200
  nist_ftc_09_asme1_rd  0.800 -> 1.000  +0.200
  nist_ctc_04_asme1_rd  0.600 -> 0.900  +0.300



'base       reader-20261102T054056Z-63e1  whole-page-v1  hash 7c7fecc1c3fe  split dev  5 case(s)\ncandidate  reader-20261102T052636Z-4cb6  vector-crop-v1  hash 48b6757c7e1d  split dev  5 case(s)\n\nmetric          base   candidate  delta   verdict\nf1 (primary)    0.757  0.739      -0.018  worse\nprecision       0.709  0.620      -0.089  worse\nrecall          0.812  0.917      +0.104  better\npartial_recall  0.633  0.867      +0.233  better\n\n5 of 5 case(s) changed case_score\n  nist_ftc_07_asme1_rd  0.724 -> 0.603  -0.121\n  nist_ctc_01_asme1_rd  0.800 -> 0.957  +0.157\n  nist_ctc_02_asme1_rc  0.500 -> 0.700  +0.200\n  nist_ftc_09_asme1_rd  0.800 -> 1.000  +0.200\n  nist_ctc_04_asme1_rd  0.600 -> 0.900  +0.300\n'